In [1]:
import matplotlib
matplotlib.rcParams.clear()
matplotlib.rcdefaults()

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# Prevent matplotlib_inline from interfering
plt.switch_backend('Agg')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import os
os.chdir('/Users/annemarie/Documents/1_TreeMort/2_Analysis/3_analysis_FORWARDS_restoration_scenarios/scripts')

print("Starting script...")

# Define SSP scenarios to process
ssp_scenarios = ['ssp126', 'ssp370']

# Define cumulative disturbance columns to plot
cumulative_columns = ['CumInsect', 'CumStorm','CumBurntArea']

# Year groupings to visualize (actual years in data: 2050, 2070, 2090)
year_groups = [2050, 2070, 2090]

lon_min, lon_max = -11, 32
lat_min, lat_max = 35, 72

output_dir = '../Figures_and_reports/disturbance_risk/'
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory ready: {output_dir}")

Starting script...
Output directory ready: ../Figures_and_reports/disturbance_risk/


In [5]:
# Track progress
total_maps = 0
skipped_maps = 0

# First, load all data to calculate shared vmax (95th percentile) across both SSPs
print("Calculating shared 95th percentile across all SSPs...")
all_data = []
for ssp in ssp_scenarios:
    input_file = f'../data/processed/{ssp}_disturbance_risk.csv'
    try:
        df = pd.read_csv(input_file)
        all_data.append(df)
    except FileNotFoundError:
        print(f"WARNING: File {input_file} not found.")
        continue

df_combined = pd.concat(all_data, ignore_index=True)

# Calculate shared vmax for each disturbance agent (across all SSPs)
vmax_dict = {}
for cum_col in cumulative_columns:
    vmax_dict[cum_col] = df_combined[cum_col].quantile(0.95)
    print(f"  {cum_col}: vmax (95th percentile) = {vmax_dict[cum_col]:.4f}")

# Override with custom values
vmax_dict['CumInsect'] = 0.2
vmax_dict['CumStorm'] = 0.8
vmax_dict['CumBurntArea'] = 0.8
print(f"  Override: CumInsect={vmax_dict['CumInsect']}, CumStorm={vmax_dict['CumStorm']}, CumBurntArea={vmax_dict['CumBurntArea']}")

# Loop through SSP scenarios
for ssp in ssp_scenarios:
    print(f"\n{'='*60}")
    print(f"Processing SSP: {ssp}")
    print(f"{'='*60}")

    # Read the CSV for this SSP
    input_file = f'../data/processed/{ssp}_disturbance_risk.csv'
    print(f"Reading {input_file}...")

    try:
        df_all = pd.read_csv(input_file)
        print(f"CSV loaded successfully. Shape: {df_all.shape}")
    except FileNotFoundError:
        print(f"WARNING: File {input_file} not found. Skipping {ssp}.")
        continue

    # Loop through cumulative columns (CumInsect, CumStorm..)
    for cum_col in cumulative_columns:
        print(f"\n  Processing {cum_col}...")
        
        # Use shared vmax across all SSPs
        vmax = vmax_dict[cum_col]
            
        # Loop through year groups
        for year in year_groups:
            filename = f'{output_dir}{cum_col}_{ssp}_{year}.png'

            print(f"    > Creating map for {year}...", end='', flush=True)

            df = df_all[df_all['Year'] == year].copy()
            df = df.dropna(subset=[cum_col, 'Lon', 'Lat'])

            if len(df) == 0:
                print(f" SKIPPED (no data)")
                continue

            # Create map
            fig = plt.figure(figsize=(11, 8))
            ax = plt.axes(projection=ccrs.PlateCarree())
            ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

            ax.add_feature(cfeature.COASTLINE.with_scale('50m'))
            ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
            ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray', alpha=0.5)

            # Point sizes: scale by forest fraction
            min_size = 5
            max_size = 30
            sizes = min_size + df['FOREST'] * (max_size - min_size)

            # Scatter plot
            sc = ax.scatter(df['Lon'], df['Lat'],
                            s=sizes,
                            c=df[cum_col],
                            cmap='Reds',
                            vmin=0,
                            vmax=vmax,
                            alpha=0.85,
                            edgecolor='k', linewidth=0.25)

            # Colorbar
            cbar_label = 'Cumulative Fraction of Biomass Lost'
            cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.03, shrink=0.65)
            cbar.set_label(cbar_label, fontsize=12)
            # override settings, if Burnt area:
            if(cum_col=="CumBurntArea"):
                cbar_label = 'Cumulative Fraction of Gridcell burnt'


            # Title
            agent_display = cum_col.replace('Cum', '')
            ax.set_title(f'{agent_display} Cumulative Disturbance Loss - {ssp.upper()} - {year}',
                    fontsize=16, fontweight='bold')

            # Size legend for forest fraction
            forest_raw = df['FOREST'].dropna().values
            if len(forest_raw) > 0:
                legend_vals = np.linspace(forest_raw.min(), forest_raw.max(), 4)
                legend_sizes = min_size + legend_vals * (max_size - min_size)

                for val, s in zip(legend_vals, legend_sizes):
                    ax.scatter([], [], s=s, color='gray', alpha=0.7, label=f'{val:.2f}')

                legend = ax.legend(scatterpoints=1, frameon=True, title='Forest Fraction',
                                loc='lower right', fontsize=10)
                legend.get_title().set_fontsize(11)

            # Save
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()

            total_maps += 1
            print(f" DONE")

print(f"\n{'='*60}")
print(f"Complete! Created: {total_maps} maps, Skipped: {skipped_maps} existing")
print(f"{'='*60}")

Calculating shared 95th percentile across all SSPs...
  CumInsect: vmax (95th percentile) = 0.0397
  CumStorm: vmax (95th percentile) = 0.2108
  CumBurntArea: vmax (95th percentile) = 0.2070
  Override: CumInsect=0.2, CumStorm=0.8, CumBurntArea=0.8

Processing SSP: ssp126
Reading ../data/processed/ssp126_disturbance_risk.csv...
CSV loaded successfully. Shape: (8054, 7)

  Processing CumInsect...
    > Creating map for 2050... DONE
    > Creating map for 2070... DONE
    > Creating map for 2090... DONE

  Processing CumStorm...
    > Creating map for 2050... DONE
    > Creating map for 2070... DONE
    > Creating map for 2090... DONE

  Processing CumBurntArea...
    > Creating map for 2050... DONE
    > Creating map for 2070... DONE
    > Creating map for 2090... DONE

Processing SSP: ssp370
Reading ../data/processed/ssp370_disturbance_risk.csv...
CSV loaded successfully. Shape: (8052, 7)

  Processing CumInsect...
    > Creating map for 2050... DONE
    > Creating map for 2070... DONE

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import os
os.chdir('/Users/annemarie/Documents/1_TreeMort/2_Analysis/3_analysis_FORWARDS_restoration_scenarios/scripts')

print("Starting script for stakeholder meetings...")

# Define SSP scenarios to process
ssp_scenarios = ['ssp126', 'ssp370']

# Define cumulative disturbance columns to plot
cumulative_columns = ['CumInsect', 'CumStorm','CumBurntArea']

# Year and regio groupings to visualize (actual years in data: 2050, 2070, 2090)
year_groups = [2050, 2070, 2090]
regions = ["Nordics","Italy","UK","Switzerland"]


output_dir = '../Figures_and_reports/disturbance_risk_stakeholder_meeting/'
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory ready: {output_dir}")

Starting script for stakeholder meetings...
Output directory ready: ../Figures_and_reports/disturbance_risk_stakeholder_meeting/


In [64]:
# Track progress - create the regional graphs for stakeholders
total_maps = 0
skipped_maps = 0

# First, load all data to calculate shared vmax (95th percentile) across both SSPs
print("Calculating shared 95th percentile across all SSPs...")
all_data = []
for ssp in ssp_scenarios:
    input_file = f'../data/processed/{ssp}_disturbance_risk.csv'
    try:
        df = pd.read_csv(input_file)
        all_data.append(df)
    except FileNotFoundError:
        print(f"WARNING: File {input_file} not found.")
        continue

df_combined = pd.concat(all_data, ignore_index=True)

# Calculate shared vmax for each disturbance agent (across all SSPs)
vmax_dict = {}
for cum_col in cumulative_columns:
    vmax_dict[cum_col] = df_combined[cum_col].quantile(0.95)
    print(f"  {cum_col}: vmax (95th percentile) = {vmax_dict[cum_col]:.4f}")

# Override with custom values
vmax_dict['CumInsect'] = 0.2
vmax_dict['CumStorm'] = 0.8
vmax_dict['CumBurntArea'] = 0.8
print(f"  Override: CumInsect={vmax_dict['CumInsect']}, CumStorm={vmax_dict['CumStorm']}, CumBurntArea={vmax_dict['CumBurntArea']}")

# Loop through SSP scenarios
for ssp in ssp_scenarios:
    print(f"\n{'='*60}")
    print(f"Processing SSP: {ssp}")
    print(f"{'='*60}")

    # Read the CSV for this SSP
    input_file = f'../data/processed/{ssp}_disturbance_risk.csv'
    print(f"Reading {input_file}...")

    try:
        df_all = pd.read_csv(input_file)
        print(f"CSV loaded successfully. Shape: {df_all.shape}")
    except FileNotFoundError:
        print(f"WARNING: File {input_file} not found. Skipping {ssp}.")
        continue

    # Loop through cumulative columns (CumInsect, CumStorm..)
    for cum_col in cumulative_columns:
        print(f"\n  Processing {cum_col}...") 
        
        # Use shared vmax across all SSPs
        vmax = vmax_dict[cum_col]
            
        # Loop through year groups
        for year in year_groups:
            for region in regions:
                if region == "Nordics":   #["Nordics","Italy","UK","Switzerland"]
                    lon_min, lon_max = 10, 32
                    lat_min, lat_max = 52, 72
                if region == "Italy":
                    lon_min, lon_max = 5, 20
                    lat_min, lat_max = 35, 48
                if region == "UK":
                    lon_min, lon_max = -11, 2
                    lat_min, lat_max = 50, 60
                if region == "Switzerland":
                   lon_min, lon_max = 5, 15
                   lat_min, lat_max = 44, 50

                filename = f'{output_dir}{region}{cum_col}_{ssp}_{year}.png'

                print(f"    > Creating map for {year}{region}...", end='', flush=True)

                df = df_all[df_all['Year'] == year].copy()
                df = df.dropna(subset=[cum_col, 'Lon', 'Lat'])

                if len(df) == 0:
                    print(f" SKIPPED (no data)")
                    continue

                # Create map
                fig = plt.figure(figsize=(11, 8))
                ax = plt.axes(projection=ccrs.PlateCarree())
                ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

                ax.add_feature(cfeature.COASTLINE.with_scale('50m'))
                ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
                ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray', alpha=0.5)

                # Lat/lon gridlines with axis labels
                gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray',
                                  alpha=0.5, linestyle='--')
                gl.top_labels = False
                gl.right_labels = False
                gl.xlabel_style = {'size': 10}
                gl.ylabel_style = {'size': 10}

                # Point sizes: scale by forest fraction
                # adjust to regional plot sizes
                # Full European extent
                full_lon_range = 32 - (-11)  # = 43°
                full_lat_range = 72 - 35     # = 37°

                # Your regional extent (adjust these)
                region_lon_range = lon_max - lon_min
                region_lat_range = lat_max - lat_min

                # Scale factor (linear, not area, since dot size is in pt²)
                scale = np.sqrt((full_lon_range * full_lat_range) /
                                (region_lon_range * region_lat_range))*1.8

                max_size = 30 * scale
                min_size = 5 * scale
                sizes = min_size + df['FOREST'] * (max_size - min_size)

                # Scatter plot
                sc = ax.scatter(df['Lon'], df['Lat'],
                                s=sizes,
                                c=df[cum_col],
                                cmap='Reds',
                                vmin=0,
                                vmax=vmax,
                                alpha=0.85,
                                edgecolor='k', linewidth=0.25)

                # Colorbar
                cbar_label = 'Cumulative Fraction of Biomass Lost'
                # override settings, if Burnt area:
                if(cum_col=="CumBurntArea"):
                    cbar_label = 'Cumulative Fraction of Gridcell burnt'
                cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.03, shrink=0.65)
                cbar.set_label(cbar_label, fontsize=12)


                # Title
                agent_display = cum_col.replace('Cum', '')
                ax.set_title(f'{agent_display} Cumulative Disturbance Loss - {ssp.upper()} - {year}',
                        fontsize=16, fontweight='bold')

                # Size legend for forest fraction
                forest_raw = df['FOREST'].dropna().values
                if len(forest_raw) > 0:
                    legend_vals = np.linspace(forest_raw.min(), forest_raw.max(), 4)
                    legend_sizes = min_size + legend_vals * (max_size - min_size)

                    for val, s in zip(legend_vals, legend_sizes):
                        ax.scatter([], [], s=s, color='gray', alpha=0.7, label=f'{val:.2f}')

                    legend = ax.legend(scatterpoints=1, frameon=True, title='Forest Fraction',
                                    loc='lower right', fontsize=10)
                    legend.get_title().set_fontsize(11)

                # Save
                plt.savefig(filename, dpi=300, bbox_inches='tight')
                plt.close()

                total_maps += 1
                print(f" DONE")

print(f"\n{'='*60}")
print(f"Complete! Created: {total_maps} maps, Skipped: {skipped_maps} existing")
print(f"{'='*60}")

Calculating shared 95th percentile across all SSPs...
  CumInsect: vmax (95th percentile) = 0.0397
  CumStorm: vmax (95th percentile) = 0.2108
  CumBurntArea: vmax (95th percentile) = 0.2070
  Override: CumInsect=0.2, CumStorm=0.8, CumBurntArea=0.8

Processing SSP: ssp126
Reading ../data/processed/ssp126_disturbance_risk.csv...
CSV loaded successfully. Shape: (8054, 7)

  Processing CumInsect...
    > Creating map for 2050Nordics... DONE
    > Creating map for 2050Italy... DONE
    > Creating map for 2050UK... DONE
    > Creating map for 2050Switzerland... DONE
    > Creating map for 2070Nordics... DONE
    > Creating map for 2070Italy... DONE
    > Creating map for 2070UK... DONE
    > Creating map for 2070Switzerland... DONE
    > Creating map for 2090Nordics... DONE
    > Creating map for 2090Italy... DONE
    > Creating map for 2090UK... DONE
    > Creating map for 2090Switzerland... DONE

  Processing CumStorm...
    > Creating map for 2050Nordics... DONE
    > Creating map for 20

In [100]:
# Track progress - create the regional graphs for stakeholders
# Layout: 2 rows (ssp126, ssp370) x 3 cols (Insect, Storm, BurntArea)
total_maps = 0

# Column order: Insect first, Wind (Storm) second, Fire (BurntArea) third
col_order = ['CumInsect', 'CumStorm', 'CumBurntArea']
col_titles = {'CumInsect': 'Insect', 'CumStorm': 'Wind', 'CumBurntArea': 'Fire'}
row_order = ['ssp126', 'ssp370']
# Year and region  groupings to visualize (actual years in data: 2050, 2070, 2090)
year_groups = [2050, 2070, 2090]
regions = ["Switzerland"]#,"Italy","UK","Switzerland"] Nordics

# First, load all data to calculate shared vmax (95th percentile) across both SSPs
print("Calculating shared 95th percentile across all SSPs...")
all_data = {}
for ssp in ssp_scenarios:
    input_file = f'../data/processed/{ssp}_disturbance_risk.csv'
    try:
        all_data[ssp] = pd.read_csv(input_file)
        print(f"  {ssp}: {all_data[ssp].shape}")
    except FileNotFoundError:
        print(f"WARNING: File {input_file} not found.")
        continue

df_combined = pd.concat(all_data.values(), ignore_index=True)

# Shared vmax across both SSPs
vmax_dict = {}
for cum_col in cumulative_columns:
    vmax_dict[cum_col] = df_combined[cum_col].quantile(0.95)
    print(f"  {cum_col}: vmax (95th percentile) = {vmax_dict[cum_col]:.4f}")

# Override with custom values
vmax_dict['CumInsect'] = 0.2
vmax_dict['CumStorm'] = 0.8
vmax_dict['CumBurntArea'] = 0.8

# Loop through regions and years
for region in regions:
    if region == "Nordics":
        lon_min, lon_max = 10, 32
        lat_min, lat_max = 52, 72
    elif region == "Italy":
        lon_min, lon_max = 5, 20
        lat_min, lat_max = 35, 48
    elif region == "UK":
        lon_min, lon_max = -11, 2
        lat_min, lat_max = 50, 60
    elif region == "Switzerland":
        lon_min, lon_max = 4, 15
        lat_min, lat_max = 42, 50

    # Point size scaling for this region
    full_lon_range = 32 - (-11)
    full_lat_range = 72 - 35
    region_lon_range = lon_max - lon_min
    region_lat_range = lat_max - lat_min
    scale =np.sqrt((full_lon_range * full_lat_range) /
                    (region_lon_range * region_lat_range))#,1.0,3.0) 
    #the above didn't really seem to work on the fly. hardcoding the sizes:
    if region == "Nordics":
        scale = 0.7
    if region == "Italy":
        scale = 1.8
    max_size = 30 * scale
    min_size = 5 * scale

    for year in year_groups:
        filename = f'{output_dir}{region}_{year}_panel.png'
        print(f"  Creating {region} {year} panel...", end='', flush=True)

        fig, axes = plt.subplots(2, 3, figsize=(16, 7),
                                 subplot_kw={'projection': ccrs.PlateCarree()})
        fig.suptitle(f'{region} - Cumulative Disturbance Loss - {year}',
                     fontsize=16, fontweight='bold', y=0.97)

        for row_idx, ssp in enumerate(row_order):
            df_all = all_data[ssp]
            df = df_all[df_all['Year'] == year].copy()

            for col_idx, cum_col in enumerate(col_order):
                ax = axes[row_idx, col_idx]
                ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

                ax.add_feature(cfeature.COASTLINE.with_scale('50m'))
                ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
                ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray', alpha=0.5)

                # Gridlines with labels only on left and bottom edges
                gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray',
                                  alpha=0.5, linestyle='--')
                gl.top_labels = False
                gl.right_labels = False
                gl.left_labels = (col_idx == 0)
                gl.bottom_labels = (row_idx == 1)
                gl.xlabel_style = {'size': 8}
                gl.ylabel_style = {'size': 8}

                df_plot = df.dropna(subset=[cum_col, 'Lon', 'Lat'])
                vmax = vmax_dict[cum_col]
                sizes = min_size + df_plot['FOREST'] * (max_size - min_size)

                sc = ax.scatter(df_plot['Lon'], df_plot['Lat'],
                                s=sizes,
                                c=df_plot[cum_col],
                                cmap='Reds',
                                vmin=0,
                                vmax=vmax,
                                alpha=0.85,
                                edgecolor='k', linewidth=0.25)

                # Colorbar
                if cum_col == 'CumBurntArea':
                    cbar_label = 'Cum. Fraction Gridcell Burnt'
                else:
                    cbar_label = 'Cum. Fraction Biomass Lost'
                cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.03, shrink=0.7)
                cbar.set_label(cbar_label, fontsize=8)
                cbar.ax.tick_params(labelsize=7)

                # Column title (top row only)
                if row_idx == 0:
                    ax.set_title(col_titles[cum_col], fontsize=12, fontweight='bold')

                # Row label (SSP, left column only)
                if col_idx == 0:
                    ax.text(-0.22, 0.5, ssp.upper(), transform=ax.transAxes,
                            fontsize=11, fontweight='bold', va='center', rotation=90)
                
                # Forest fraction size legend — top-left subplot only
                if row_idx == 0 and col_idx == 0:
                    forest_raw = df_plot['FOREST'].values
                    legend_vals = np.linspace(forest_raw.min(), forest_raw.max(), 4)
                    legend_sizes = min_size + legend_vals * (max_size - min_size)
                    for val, s in zip(legend_vals, legend_sizes):
                        ax.scatter([], [], s=s, color='gray', alpha=0.7, label=f'{val:.2f}')
                    leg = ax.legend(scatterpoints=1, frameon=True, title='Forest\nFraction',
                                    loc='upper left', fontsize=8)
                    leg.get_title().set_fontsize(9)

        plt.subplots_adjust(hspace=0.05, left=0.2,wspace = 0.01,top=0.83)
        plt.savefig(filename, dpi=200, bbox_inches='tight')
        plt.close()

        total_maps += 1
        print(f" DONE")

print(f"\n{'='*60}")
print(f"Complete! Created: {total_maps} panel figures")
print(f"{'='*60}")

Calculating shared 95th percentile across all SSPs...
  ssp126: (8054, 7)
  ssp370: (8052, 7)
  CumInsect: vmax (95th percentile) = 0.0397
  CumStorm: vmax (95th percentile) = 0.2108
  CumBurntArea: vmax (95th percentile) = 0.2070
  Creating Switzerland 2050 panel... DONE
  Creating Switzerland 2070 panel... DONE
  Creating Switzerland 2090 panel... DONE

Complete! Created: 3 panel figures


In [93]:
#also create (regional) difference plots for the 4 periods
# Script to plot LPJ-GUESS output data processed with FORWARDS_analysis.Rmd chunk Risk_analysis_stakeholders_carbon_loss
# Feb 2026
# Annemarie Eckes-Shephard
# Claude 
# based on example script from Haoming Zhong.


import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import os

# Year groupings to visualize (actual years in data: 2050, 2070, 2090)
year_groups = [2050, 2070, 2090]

print("Starting difference plot script...")

# Define what columns to plot
# Format: {plot_name: column_name}
plot_configs = {
    'ssp126': {
        'Vegetation_carbon 2050-2030': 'ssp126VegCdiff_2050_2030',
        'Vegetation_carbon 2070-2050': 'ssp126VegCdiff_2070_2050',
        'Vegetation_carbon 2090-2070': 'ssp126VegCdiff_2090_2070',
        'Vegetation_carbon 2090-2030': 'ssp126VegCdiff_2090_2030',
        'Ecosystem_carbon 2050-2030': 'ssp126EcosystemCdiff_2050_2030',
        'Ecosystem_carbon 2070-2050': 'ssp126EcosystemCdiff_2070_2050',
        'Ecosystem_carbon 2090-2070': 'ssp126EcosystemCdiff_2090_2070',
        'Ecosystem_carbon 2090-2030': 'ssp126EcosystemCdiff_2090_2030'
       
    },
    'ssp370': {
        'Vegetation_carbon 2050-2030': 'ssp370VegCdiff_2050_2030',
        'Vegetation_carbon 2070-2050': 'ssp370VegCdiff_2070_2050',
        'Vegetation_carbon 2090-2070': 'ssp370VegCdiff_2090_2070',
        'Vegetation_carbon 2090-2030': 'sssp370VegCdiff_2090_2030',
        'Ecosystem_carbon 2050-2030': 'sssp370EcosystemCdiff_2050_2030',
        'Ecosystem_carbon 2070-2050': 'ssp370EcosystemCdiff_2070_2050',
        'Ecosystem_carbon 2090-2070': 'ssp370EcosystemCdiff_2090_2070',
        'Ecosystem_carbon 2090-2030': 'ssp370EcosystemCdiff_2090_2030'
    }
}

lon_min, lon_max = -11, 32
lat_min, lat_max = 35, 72

output_dir = '../Figures_and_reports/disturbance_risk_stakeholder_meeting/'
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory ready: {output_dir}")

# Read the single CSV file that contains all data
input_file = '../data/processed/Forest_and_ecosystem_carbon_loss_data_diff_selected_years.csv'
print(f"Reading {input_file}...")

try:
    df = pd.read_csv(input_file)
    print(f"CSV loaded successfully. Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
except FileNotFoundError:
    print(f"ERROR: File {input_file} not found. Exiting.")
    exit(1)

total_maps = 0

# Create plots for each SSP and carbon type
for ssp, carbon_types in plot_configs.items():
    print(f"\n{'='*60}")
    print(f"Processing SSP: {ssp}")
    print(f"{'='*60}")

    for carbon_name, column_name in carbon_types.items():
        print(f"\n>>> Processing {carbon_name}...")

        # Check if column exists
        if column_name not in df.columns:
            print(f"     WARNING: Column '{column_name}' not found. Skipping.")
            print(f"     Available columns: {df.columns.tolist()}")
            continue

        filename = f'{output_dir}{carbon_name}_{ssp}_difference.png'

        if os.path.exists(filename):
            print(f"    File already exists, skipping")
            continue

        print(f"    Creating difference map...", end='', flush=True)

        # Prepare data
        df_plot = df[['Lon', 'Lat', col_name, 'FOREST']].dropna()
        df_plot = df_plot.rename(columns={col_name: 'diff'})

        # Drop NA values
        df_plot = df_plot.dropna()

        if len(df_plot) == 0:
            print(f" SKIPPED (no valid data)")
            continue

        # Create map
        fig = plt.figure(figsize=(11, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.COASTLINE.with_scale('50m'))
        ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
        ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray', alpha=0.5)

        max_size = 30
        min_size = 5
        # Point sizes scaled by forest fraction at regional dot density
        sizes = min_size + df_plot['FOREST'] * (max_size - min_size)


        # Use red-white-blue diverging colormap
        # Symmetric limits around zero (as in R code)
        if carbon_name in ['Ecosystem_carbon 2090-2030','Vegetation_carbon 2090-2030']:#['ssp126EcosystemCdiff_2090_2030', 'ssp370cosystemCdiff_2090_2030','sssp370VegCdiff_2090_2030','ssp126VegCdiff_2090_2030']:
            vmax = 8
            vmin = -8
        else:
            vmax = 4
            vmin = -4

        # Scatter plot with diverging colormap
        sc = ax.scatter(df_plot['Lon'], df_plot['Lat'],
                       s=sizes,
                       c=df_plot['diff'],
                       cmap='RdBu',  # Red for negative (loss), Blue for positive (gain)
                       vmin=vmin,
                       vmax=vmax,
                       alpha=0.85,
                       edgecolor='k', linewidth=0.25)

        # Colorbar
        cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.03, shrink=0.65,
                           extend='both')  # Show arrows for values beyond limits
        cbar.set_label('Δ Carbon (kgC/m²)', fontsize=12)
        

        # Title
        carbon_label = carbon_name.replace('_', ' ').title()
        ax.set_title(f'{carbon_label} Change - {ssp.upper()}',
                   fontsize=16, fontweight='bold')

        # Save
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()

        total_maps += 1
        print(f" DONE")

print(f"\n{'='*60}")
print(f"Script complete! Total difference maps created: {total_maps}")
print(f"{'='*60}")



Starting difference plot script...
Output directory ready: ../Figures_and_reports/disturbance_risk_stakeholder_meeting/
Reading ../data/processed/Forest_and_ecosystem_carbon_loss_data_diff_selected_years.csv...
CSV loaded successfully. Shape: (2686, 23)
Columns: ['Lon', 'Lat', '2030', '2050', '2070', '2090', 'ssp126VegCdiff_2050_2030', 'ssp126VegCdiff_2070_2050', 'ssp126VegCdiff_2090_2070', 'ssp126VegCdiff_2090_2030', 'ssp370VegCdiff_2050_2030', 'ssp370VegCdiff_2070_2050', 'ssp370VegCdiff_2090_2070', 'ssp370VegCdiff_2090_2030', 'ssp126EcosystemCdiff_2050_2030', 'ssp126EcosystemCdiff_2070_2050', 'ssp126EcosystemCdiff_2090_2070', 'ssp126EcosystemCdiff_2090_2030', 'ssp370EcosystemCdiff_2050_2030', 'ssp370EcosystemCdiff_2070_2050', 'ssp370EcosystemCdiff_2090_2070', 'ssp370EcosystemCdiff_2090_2030', 'FOREST']

Processing SSP: ssp126

>>> Processing Vegetation_carbon 2050-2030...
    File already exists, skipping

>>> Processing Vegetation_carbon 2070-2050...
    File already exists, skippin

In [94]:
# Regional carbon difference panels
# Layout: 2 rows (ssp126, ssp370) x 2 cols (Vegetation C, Ecosystem C)
# One panel per region x year_group period — mirrors the disturbance panel structure
total_maps = 0

ssp_scenarios = ['ssp126', 'ssp370']
regions = ["Nordics", "Italy", "UK", "Switzerland"]

# Four difference periods: three successive decades + cumulative 2090-2030
year_groups = ['2050_2030', '2070_2050', '2090_2070', '2090_2030']
year_labels  = {
    '2050_2030': '2050-2030',
    '2070_2050': '2070-2050',
    '2090_2070': '2090-2070',
    '2090_2030': '2090-2030'
}

# Carbon column prefixes (appended after ssp name, before _year_key)
col_order  = ['VegCdiff', 'EcosystemCdiff']
col_titles = {'VegCdiff': 'Vegetation Carbon', 'EcosystemCdiff': 'Ecosystem Carbon'}

# Wider colour range for the long cumulative period
vrange = {'2090_2030': (-8, 8)}
vrange_default = (-4, 4)

output_dir = '../Figures_and_reports/disturbance_risk_stakeholder_meeting/'
os.makedirs(output_dir, exist_ok=True)

# Read data (FOREST column now present — fraction of gridcell covered by forest)
input_file = '../data/processed/Forest_and_ecosystem_carbon_loss_data_diff_selected_years.csv'
print(f"Reading {input_file}...")
try:
    df = pd.read_csv(input_file)
    print(f"CSV loaded. Shape: {df.shape}")
except FileNotFoundError:
    print(f"ERROR: File {input_file} not found. Exiting.")
    exit(1)

for region in regions:
    if region == "Nordics":
        lon_min, lon_max = 10, 32
        lat_min, lat_max = 52, 72
    elif region == "Italy":
        lon_min, lon_max = 5, 20
        lat_min, lat_max = 35, 48
    elif region == "UK":
        lon_min, lon_max = -11, 2
        lat_min, lat_max = 50, 60
    elif region == "Switzerland":
        lon_min, lon_max = 4, 15
        lat_min, lat_max = 42, 50

    # Point size scaling: match apparent dot area to what it would be on the pan-European map
    full_lon_range = 32 - (-11)
    full_lat_range = 72 - 35
    region_lon_range = lon_max - lon_min
    region_lat_range = lat_max - lat_min
    scale = np.sqrt((full_lon_range * full_lat_range) /
                    (region_lon_range * region_lat_range))
    if region == "Nordics":
        scale = 0.7
    if region == "Italy":
        scale = 1.8
    max_size = 30 * scale
    min_size = 5 * scale

    for year_key in year_groups:
        year_label = year_labels[year_key]
        vmin, vmax = vrange.get(year_key, vrange_default)

        filename = f'{output_dir}{region}_{year_label}_carbon_panel.png'
        print(f"  Creating {region} {year_label} carbon panel...", end='', flush=True)

        fig, axes = plt.subplots(2, 2, figsize=(12, 7),
                                 subplot_kw={'projection': ccrs.PlateCarree()})
        fig.suptitle(f'{region} - Carbon Change - {year_label}',
                     fontsize=16, fontweight='bold', y=0.97)

        for row_idx, ssp in enumerate(ssp_scenarios):
            for col_idx, carbon_col in enumerate(col_order):
                col_name = f'{ssp}{carbon_col}_{year_key}'
                ax = axes[row_idx, col_idx]
                ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

                ax.add_feature(cfeature.COASTLINE.with_scale('50m'))
                ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
                ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray', alpha=0.5)

                gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray',
                                  alpha=0.5, linestyle='--')
                gl.top_labels = False
                gl.right_labels = False
                gl.left_labels = (col_idx == 0)
                gl.bottom_labels = (row_idx == 1)
                gl.xlabel_style = {'size': 8}
                gl.ylabel_style = {'size': 8}

                if col_name not in df.columns:
                    print(f"\n     WARNING: '{col_name}' not found, skipping subplot.")
                    ax.set_title('(no data)', fontsize=9)
                    continue

                df_plot = df[['Lon', 'Lat', col_name, 'FOREST']].dropna()
                df_plot = df_plot.rename(columns={col_name: 'diff'})

                # Point sizes scaled by forest fraction at regional dot density
                sizes = min_size + df_plot['FOREST'] * (max_size - min_size)

                sc = ax.scatter(df_plot['Lon'], df_plot['Lat'],
                                s=sizes,
                                c=df_plot['diff'],
                                cmap='RdBu',
                                vmin=vmin,
                                vmax=vmax,
                                alpha=0.85,
                                edgecolor='k', linewidth=0.25)

                cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.03,
                                    shrink=0.7, extend='both')
                cbar.set_label('Δ Carbon (kgC/m²)', fontsize=8)
                cbar.ax.tick_params(labelsize=7)

                # Column title on top row only
                if row_idx == 0:
                    ax.set_title(col_titles[carbon_col], fontsize=12, fontweight='bold')

                # SSP row label on left column only
                if col_idx == 0:
                    ax.text(-0.22, 0.5, ssp.upper(), transform=ax.transAxes,
                            fontsize=11, fontweight='bold', va='center', rotation=90)

                # Forest fraction size legend — top-left subplot only
                if row_idx == 0 and col_idx == 0:
                    forest_raw = df_plot['FOREST'].values
                    legend_vals = np.linspace(forest_raw.min(), forest_raw.max(), 4)
                    legend_sizes = min_size + legend_vals * (max_size - min_size)
                    for val, s in zip(legend_vals, legend_sizes):
                        ax.scatter([], [], s=s, color='gray', alpha=0.7, label=f'{val:.2f}')
                    leg = ax.legend(scatterpoints=1, frameon=True, title='Forest\nFraction',
                                    loc='upper left', fontsize=8)
                    leg.get_title().set_fontsize(9)

        plt.subplots_adjust(hspace=0.05, left=0.2, wspace=0.15, top=0.83)
        plt.savefig(filename, dpi=200, bbox_inches='tight')
        plt.close()

        total_maps += 1
        print(f" DONE")

print(f"\n{'='*60}")
print(f"Complete! Created: {total_maps} regional carbon panel figures")
print(f"{'='*60}")

Reading ../data/processed/Forest_and_ecosystem_carbon_loss_data_diff_selected_years.csv...
CSV loaded. Shape: (2686, 23)
  Creating Nordics 2050-2030 carbon panel... DONE
  Creating Nordics 2070-2050 carbon panel... DONE
  Creating Nordics 2090-2070 carbon panel... DONE
  Creating Nordics 2090-2030 carbon panel... DONE
  Creating Italy 2050-2030 carbon panel... DONE
  Creating Italy 2070-2050 carbon panel... DONE
  Creating Italy 2090-2070 carbon panel... DONE
  Creating Italy 2090-2030 carbon panel... DONE
  Creating UK 2050-2030 carbon panel... DONE
  Creating UK 2070-2050 carbon panel... DONE
  Creating UK 2090-2070 carbon panel... DONE
  Creating UK 2090-2030 carbon panel... DONE
  Creating Switzerland 2050-2030 carbon panel... DONE
  Creating Switzerland 2070-2050 carbon panel... DONE
  Creating Switzerland 2090-2070 carbon panel... DONE
  Creating Switzerland 2090-2030 carbon panel... DONE

Complete! Created: 16 regional carbon panel figures
